In [52]:
import pandas as pd

df = pd.read_csv(r'E:\Pyhton code\Data Sets\archive\100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [53]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace("?","")
    text = text.replace("'","")
    return text.split()

In [54]:
tokenize("My name is Suyash")

['my', 'name', 'is', 'suyash']

In [55]:
# vocab
vocab = {"<UNK>":0}
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens  = tokenized_question + tokenized_answer 
    print(merged_tokens)
    
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)
    

In [56]:
df.apply(build_vocab, axis=1)

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '19

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [57]:
len(vocab)

324

In [58]:
# Convert words to numerical indices
def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [59]:
text_to_indices("What is my name?", vocab)

[1, 2, 0, 0]

In [60]:
import torch
from torch.utils.data import Dataset, DataLoader

In [61]:
class QADataset(Dataset):
    
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
        
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numerical_question = text_to_indices (self.df.iloc[index]['question'], self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [62]:
dataset = QADataset(df, vocab)

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

In [63]:
for question, answer in dataloader:
    print(question, answer)

tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([[128]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[ 1,  2,  3, 17, 18, 19, 20, 21, 22]]) tensor([[23]])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([[259]])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([[205]])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([[188]])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([[298]])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([[295]])
tensor([[ 10,  29, 130, 131]]) tensor([[132]])
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([[220]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([[58]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([[173]])
tensor([[ 10,  75, 111]]) tensor([[112]])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([[85]])
tensor([[  1,   2,   3, 163, 164, 165,  83,  84]]) tensor([[166]])
tensor

In [64]:
# rnn architecture
import torch.nn as nn

In [65]:
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim=50
        )

        self.rnn = nn.RNN(
            input_size=50,
            hidden_size=64,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            vocab_size
        )

    def forward(self, question):
        embedded_question = self.embedding(question)

        output, hidden = self.rnn(embedded_question)

        final_hidden = hidden[-1]

        output = self.fc(final_hidden)

        return output

In [66]:
learning_rate = 0.001
epochs = 20

In [67]:
model = SimpleRNN(len(vocab))

In [68]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), learning_rate)

In [69]:
question, answer = next(iter(dataloader))

print("Question shape:", question.shape)
print("Answer shape:", answer.shape)

output = model(question)

print("Output shape:", output.shape)

Question shape: torch.Size([1, 8])
Answer shape: torch.Size([1, 1])
Output shape: torch.Size([1, 324])


In [70]:
# training loop 
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        #forward pass
        output = model(question)
        
        #loss
        loss = criterion(output, answer.squeeze(1))
        
        # gradient
        loss.backward()
        
        # update
        optimizer.step()
        
        total_loss = total_loss + loss.item()
        
    print(f"Epoch: {epoch+1}, Loss:{total_loss:4f}")

Epoch: 1, Loss:523.550576
Epoch: 2, Loss:451.159595
Epoch: 3, Loss:372.346170
Epoch: 4, Loss:312.891388
Epoch: 5, Loss:261.869181
Epoch: 6, Loss:213.475538
Epoch: 7, Loss:169.253670
Epoch: 8, Loss:131.680043
Epoch: 9, Loss:100.749495
Epoch: 10, Loss:76.915715
Epoch: 11, Loss:59.258531
Epoch: 12, Loss:46.199650
Epoch: 13, Loss:36.973096
Epoch: 14, Loss:29.713557
Epoch: 15, Loss:24.621527
Epoch: 16, Loss:20.695039
Epoch: 17, Loss:17.639167
Epoch: 18, Loss:15.312891
Epoch: 19, Loss:13.258350
Epoch: 20, Loss:11.544675


In [76]:
def predict(model, question, threshold=0.5):
    #convert question to numbers
    numerical_question = text_to_indices(question, vocab)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    output = model(question_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)
    value, index = torch.max(probs, dim=1)
    
    if value < threshold:
        print("I don't know")
        
    print(list(vocab.keys())[index])

In [77]:
predict(model, "What is the capital of france")

paris
